# Reconocimiento de dígitos escritos a mano con una CNN

Esta versión reemplaza el clasificador SVM por una **red neuronal convolucional (CNN)** entrenada con MNIST. Reconoce dígitos del 0 al 9 a partir de un dibujo hecho con el ratón, lápiz o pantalla táctil.

## Ejecutar en orden

1. Ejecuta las celdas en orden; instala TensorFlow y descarga MNIST en el primer entrenamiento.
2. Entrena y guarda la CNN.
3. Genera los archivos de la API y la interfaz web.
4. Ejecuta la prueba de la API. Debe mostrar la exactitud obtenida y más de una clase predicha.
5. Inicia la nueva aplicación y abre **http://127.0.0.1:5001**.

**Esta versión crea una carpeta independiente `proyecto_digitos_cnn` y utiliza el puerto 5001**, para no reutilizar accidentalmente el backend o modelo SVM anterior que pudiera seguir ejecutándose en el puerto 5000.

El conjunto MNIST contiene dígitos manuscritos de 28×28 píxeles. El preprocesamiento de la interfaz convierte el fondo blanco y el trazo oscuro al formato que espera la CNN: fondo negro y dígito claro.

In [ ]:
%pip install -q tensorflow flask pillow scipy numpy

from pathlib import Path
import os

ROOT = Path.cwd()
PROJECT_DIR = ROOT / "proyecto_digitos_cnn"
BACKEND_DIR = PROJECT_DIR / "backend"
FRONTEND_DIR = PROJECT_DIR / "frontend"
TEMPLATES_DIR = FRONTEND_DIR / "templates"
STATIC_DIR = FRONTEND_DIR / "static"
for folder in (BACKEND_DIR, TEMPLATES_DIR, STATIC_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Proyecto CNN:", PROJECT_DIR.resolve())
print("Se abrirá en: http://127.0.0.1:5001")

## 1. Preprocesamiento de las imágenes

El mismo preprocesamiento se usa en las predicciones de la página y en las pruebas. Recorta el trazo, conserva la proporción, lo escala a un área de 20×20 píxeles y lo centra en un lienzo de 28×28, con el fondo negro que espera MNIST.

In [ ]:
%%writefile proyecto_digitos_cnn/backend/preprocess.py
"""Convierte un dibujo de fondo blanco y trazo oscuro al formato MNIST."""
import numpy as np
from PIL import Image
from scipy import ndimage

OUT_SIZE = 28
DIGIT_SIZE = 20


def to_model_image(pixels):
    """Recibe una imagen en escala de grises (255=fondo blanco, 0=trazo negro).

    Devuelve una matriz float32 (28, 28, 1), con fondo negro, dígito claro,
    intensidades en [0, 1]. Devuelve None si el lienzo está vacío.
    """
    gray = np.asarray(pixels, dtype=np.float32)
    if gray.ndim == 3:
        gray = gray[..., :3].mean(axis=2)
    if gray.ndim != 2 or gray.size == 0:
        raise ValueError("La imagen debe ser una matriz de escala de grises.")

    ink = np.clip(255.0 - gray, 0, 255)
    peak = float(ink.max())
    if peak < 25:
        return None

    # Ignorar el antialiasing muy tenue para encontrar el contorno del dígito.
    threshold = max(20.0, 0.16 * peak)
    ys, xs = np.where(ink >= threshold)
    if len(xs) == 0:
        return None
    crop = ink[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

    h, w = crop.shape
    scale = DIGIT_SIZE / max(h, w)
    new_w = max(1, min(DIGIT_SIZE, int(round(w * scale))))
    new_h = max(1, min(DIGIT_SIZE, int(round(h * scale))))
    crop_u8 = np.clip(crop, 0, 255).astype(np.uint8)
    resized = Image.fromarray(crop_u8).resize((new_w, new_h), Image.Resampling.LANCZOS)
    resized = np.asarray(resized, dtype=np.float32) / 255.0

    canvas = np.zeros((OUT_SIZE, OUT_SIZE), dtype=np.float32)
    top = (OUT_SIZE - new_h) // 2
    left = (OUT_SIZE - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = resized

    # Centrado fino por centro de masa, similar al formato del conjunto MNIST.
    mass = float(canvas.sum())
    if mass > 1e-6:
        yy, xx = np.indices(canvas.shape)
        cy = float((yy * canvas).sum() / mass)
        cx = float((xx * canvas).sum() / mass)
        target = (OUT_SIZE - 1) / 2.0
        canvas = ndimage.shift(
            canvas, (target - cy, target - cx), order=1,
            mode="constant", cval=0.0, prefilter=False
        )

    return np.clip(canvas, 0.0, 1.0).astype(np.float32)[..., np.newaxis]

## 2. Entrenamiento de la CNN

Se usa MNIST en lugar de depender principalmente de dibujos artificiales. La red tiene capas convolucionales para aprender bordes, curvas y combinaciones de trazos; también aplica pequeñas rotaciones y traslaciones durante el entrenamiento para tolerar variaciones al escribir.

La primera ejecución descarga el conjunto de datos y puede tardar unos minutos, según el equipo y la conexión.

In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# MNIST ya usa fondo negro y dígito claro, igual que la entrada de la CNN.
x_train = (x_train.astype("float32") / 255.0)[..., np.newaxis]
x_test = (x_test.astype("float32") / 255.0)[..., np.newaxis]
y_train = y_train.astype("int64")
y_test = y_test.astype("int64")

inputs = keras.Input(shape=(28, 28, 1), name="imagen_digito")
x = layers.RandomRotation(0.06, fill_mode="constant", fill_value=0.0, name="rotacion_aleatoria")(inputs)
x = layers.RandomTranslation(0.08, 0.08, fill_mode="constant", fill_value=0.0, name="traslacion_aleatoria")(x)
x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Dropout(0.20)(x)
x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Dropout(0.25)(x)
x = layers.Flatten()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(10, activation="softmax", name="probabilidades_0_a_9")(x)

model = keras.Model(inputs, outputs, name="CNN_MNIST")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=2, restore_best_weights=True, mode="max"
    )
]
history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=8,
    batch_size=128,
    callbacks=callbacks,
    verbose=1,
)

loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Exactitud en las 10.000 imágenes de prueba MNIST: {accuracy:.2%}")

MODEL_PATH = BACKEND_DIR / "digit_cnn.keras"
model.save(MODEL_PATH)
with open(BACKEND_DIR / "model_info.json", "w", encoding="utf-8") as f:
    json.dump({"model": "CNN-MNIST", "classes": list(range(10)), "test_accuracy": float(accuracy)}, f, indent=2)
print("Modelo CNN guardado en:", MODEL_PATH.resolve())

## 3. API Flask

La API carga explícitamente `digit_cnn.keras`, no el archivo antiguo `digit_model.joblib`. El endpoint de salud informa qué modelo cargó y la predicción devuelve la clase principal y las tres probabilidades más altas, lo que facilita diagnosticar si el modelo está confundido.

In [ ]:
%%writefile proyecto_digitos_cnn/backend/app.py
import base64
import binascii
import io
import json
import os
from pathlib import Path

import numpy as np
import tensorflow as tf
from flask import Flask, jsonify, render_template, request
from PIL import Image, UnidentifiedImageError
from preprocess import to_model_image

BASE_DIR = Path(__file__).resolve().parent
PROJECT_DIR = BASE_DIR.parent
MODEL_PATH = BASE_DIR / "digit_cnn.keras"
INFO_PATH = BASE_DIR / "model_info.json"

if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"No se encontró {MODEL_PATH}. Ejecuta primero la celda de entrenamiento CNN."
    )
model = tf.keras.models.load_model(MODEL_PATH)
model_info = json.loads(INFO_PATH.read_text(encoding="utf-8")) if INFO_PATH.is_file() else {}

app = Flask(
    __name__,
    template_folder=str(PROJECT_DIR / "frontend" / "templates"),
    static_folder=str(PROJECT_DIR / "frontend" / "static"),
)

@app.get("/")
def index():
    return render_template("index.html")

@app.get("/api/health")
def health():
    return jsonify({
        "status": "ok",
        "model": "CNN-MNIST",
        "model_path": str(MODEL_PATH.resolve()),
        "classes": list(range(10)),
        "test_accuracy": model_info.get("test_accuracy"),
    })

@app.post("/api/predict")
def predict():
    payload = request.get_json(silent=True)
    if not isinstance(payload, dict) or not isinstance(payload.get("image"), str):
        return jsonify({"error": "Envía un JSON con la propiedad 'image'."}), 400

    image_data = payload["image"]
    if image_data.startswith("data:"):
        header, sep, image_data = image_data.partition(",")
        if not sep or ";base64" not in header:
            return jsonify({"error": "La imagen debe enviarse en base64."}), 400
    if not image_data or len(image_data) > 1_000_000:
        return jsonify({"error": "La imagen está vacía o supera el tamaño permitido."}), 400

    try:
        raw = base64.b64decode(image_data, validate=True)
        with Image.open(io.BytesIO(raw)) as source:
            rgba = source.convert("RGBA")
            white_bg = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
            pixels = np.asarray(Image.alpha_composite(white_bg, rgba).convert("L"), dtype=np.uint8)
    except (binascii.Error, ValueError, UnidentifiedImageError, OSError):
        return jsonify({"error": "No se pudo leer la imagen. Envía un PNG válido."}), 400

    try:
        features = to_model_image(pixels)
    except ValueError as exc:
        return jsonify({"error": str(exc)}), 400
    if features is None:
        return jsonify({"error": "El lienzo está vacío. Dibuja un dígito primero."}), 400

    probabilities = model.predict(features[np.newaxis, ...], verbose=0)[0]
    order = np.argsort(probabilities)[::-1][:3]
    best = int(order[0])
    return jsonify({
        "digit": best,
        "confidence": float(probabilities[best]),
        "top3": [{"digit": int(i), "probability": float(probabilities[i])} for i in order],
        "model": "CNN-MNIST",
    })

if __name__ == "__main__":
    port = int(os.environ.get("DIGIT_PORT", "5001"))
    app.run(host="127.0.0.1", port=port, debug=False, use_reloader=False)

## 4. Interfaz para dibujar

La página informa de la predicción y muestra las alternativas principales. Está servida desde la misma API para evitar problemas de CORS o de rutas entre frontend y backend.

In [ ]:
%%writefile proyecto_digitos_cnn/frontend/templates/index.html
<!doctype html>
<html lang="es">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <meta name="theme-color" content="#f4f6fb">
  <title>Reconocimiento de dígitos con CNN</title>
  <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
  <script src="{{ url_for('static', filename='app.js') }}" defer></script>
</head>
<body>
  <main class="card">
    <p class="eyebrow">Red neuronal convolucional · MNIST</p>
    <h1>¿Qué número escribí?</h1>
    <p class="intro">Escribe un solo número del 0 al 9. Luego pulsa “Reconocer”.</p>
    <canvas id="drawing" width="280" height="280" aria-label="Lienzo para escribir un dígito"></canvas>
    <div class="actions">
      <button id="clear" class="secondary" type="button">Borrar</button>
      <button id="predict" type="button">Reconocer dígito</button>
    </div>
    <section class="output" aria-live="polite">
      <p id="result" class="result">Modelo CNN-MNIST listo.</p>
      <p id="alternatives" class="alternatives"></p>
    </section>
    <p class="hint">Consejo: escribe el dígito grande, centrado y sin dibujar más de un número.</p>
  </main>
</body>
</html>

In [ ]:
%%writefile proyecto_digitos_cnn/frontend/static/style.css
:root {
  color-scheme: light;
  font-family: Inter, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
  color: #172033;
  background: #f4f6fb;
}
* { box-sizing: border-box; }
body { min-height: 100vh; margin: 0; display: grid; place-items: center; padding: 24px; }
.card { width: min(100%, 480px); padding: 32px; text-align: center; background: #fff; border: 1px solid #e2e7f0; border-radius: 22px; box-shadow: 0 18px 55px #26365c12; }
.eyebrow { margin: 0 0 8px; color: #5368d5; font-size: .78rem; font-weight: 800; letter-spacing: .08em; text-transform: uppercase; }
h1 { margin: 0; font-size: clamp(1.7rem, 6vw, 2.15rem); }
.intro { margin: 12px 0 22px; color: #657087; line-height: 1.5; }
canvas { display: block; width: min(100%, 280px); height: auto; margin: auto; border: 2px solid #dce2ee; border-radius: 14px; background: #fff; cursor: crosshair; touch-action: none; }
.actions { display: flex; gap: 12px; margin-top: 18px; }
button { flex: 1; min-height: 46px; padding: 10px 14px; border: 0; border-radius: 11px; background: #5368d5; color: #fff; font: inherit; font-weight: 700; cursor: pointer; }
button:hover { background: #4055c2; }
button:focus-visible { outline: 3px solid #9ba9ff; outline-offset: 3px; }
button:disabled { cursor: wait; opacity: .65; }
button.secondary { border: 1px solid #dce2ee; background: #fff; color: #34405a; }
button.secondary:hover { background: #f5f7fb; }
.output { margin-top: 19px; padding: 14px; border-radius: 12px; background: #f6f8fc; }
.result { margin: 0; color: #26365c; font-weight: 800; }
.alternatives { margin: 7px 0 0; color: #657087; font-size: .9rem; line-height: 1.5; }
.hint { margin: 16px 0 0; color: #7a8497; font-size: .82rem; line-height: 1.45; }
@media (max-width: 420px) { .card { padding: 24px 18px; } .actions { flex-direction: column-reverse; } }

In [ ]:
%%writefile proyecto_digitos_cnn/frontend/static/app.js
const canvas = document.querySelector("#drawing");
const context = canvas.getContext("2d");
const result = document.querySelector("#result");
const alternatives = document.querySelector("#alternatives");
const predictButton = document.querySelector("#predict");
let drawing = false;
let hasInk = false;

function resetCanvas() {
  context.fillStyle = "#fff";
  context.fillRect(0, 0, canvas.width, canvas.height);
  context.strokeStyle = "#000";
  context.lineWidth = 25;
  context.lineCap = "round";
  context.lineJoin = "round";
}
resetCanvas();

function point(event) {
  const bounds = canvas.getBoundingClientRect();
  return {
    x: (event.clientX - bounds.left) * canvas.width / bounds.width,
    y: (event.clientY - bounds.top) * canvas.height / bounds.height,
  };
}
canvas.addEventListener("pointerdown", (event) => {
  drawing = true;
  hasInk = true;
  canvas.setPointerCapture(event.pointerId);
  const p = point(event);
  context.beginPath();
  context.moveTo(p.x, p.y);
  context.lineTo(p.x + 0.1, p.y + 0.1);
  context.stroke();
});
canvas.addEventListener("pointermove", (event) => {
  if (!drawing) return;
  const p = point(event);
  context.lineTo(p.x, p.y);
  context.stroke();
});
function finishStroke() { drawing = false; }
canvas.addEventListener("pointerup", finishStroke);
canvas.addEventListener("pointercancel", finishStroke);

 document.querySelector("#clear").addEventListener("click", () => {
  resetCanvas();
  hasInk = false;
  result.textContent = "Modelo CNN-MNIST listo.";
  alternatives.textContent = "";
});

predictButton.addEventListener("click", async () => {
  if (!hasInk) {
    result.textContent = "Primero escribe un dígito en el recuadro.";
    alternatives.textContent = "";
    return;
  }
  predictButton.disabled = true;
  result.textContent = "Reconociendo…";
  alternatives.textContent = "";
  try {
    const response = await fetch("/api/predict", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ image: canvas.toDataURL("image/png") }),
    });
    const data = await response.json();
    if (!response.ok) throw new Error(data.error || "No se pudo reconocer el dígito.");
    result.textContent = `Predicción: ${data.digit} · confianza ${(data.confidence * 100).toFixed(1)}%`;
    const rest = data.top3.slice(1).map(item => `${item.digit}: ${(item.probability * 100).toFixed(1)}%`);
    alternatives.textContent = `Otras opciones: ${rest.join(" · ")} · modelo ${data.model}`;
  } catch (error) {
    result.textContent = `Error: ${error.message}`;
  } finally {
    predictButton.disabled = false;
  }
});

## 5. Prueba de extremo a extremo

Esta comprobación envía al endpoint `/api/predict` imágenes del conjunto de prueba MNIST convertidas al mismo formato de lienzo que utiliza la página. Debe comprobar que el servicio responde con el modelo CNN, predice distintas clases y no intenta reconocer un lienzo vacío.

In [ ]:
import base64
import io
import sys
import numpy as np
from PIL import Image

sys.path.insert(0, str(BACKEND_DIR.resolve()))
from app import app


def as_data_url(image):
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("ascii")


def mnist_to_canvas(mnist_image):
    # MNIST: fondo negro y dígito claro. Lienzo web: fondo blanco y trazo negro.
    canvas = Image.fromarray((255 - mnist_image).astype(np.uint8), mode="L")
    return canvas.resize((280, 280), Image.Resampling.BICUBIC)

# Escoger 10 imágenes de prueba por clase (100 predicciones en total).
cases = []
for digit in range(10):
    ids = np.where(y_test == digit)[0][:10]
    cases.extend((mnist_to_canvas(x_test[i, :, :, 0] * 255).convert("L"), digit) for i in ids)

correct = 0
predictions = []
with app.test_client() as client:
    health = client.get("/api/health")
    assert health.status_code == 200, health.get_json()
    health_data = health.get_json()
    print("Servicio cargado:", health_data)
    assert health_data.get("model") == "CNN-MNIST", "Se está usando un servidor/modelo distinto del nuevo CNN."

    for image, expected in cases:
        response = client.post("/api/predict", json={"image": as_data_url(image)})
        assert response.status_code == 200, response.get_json()
        payload = response.get_json()
        predictions.append(payload["digit"])
        correct += int(payload["digit"] == expected)

    blank = as_data_url(Image.new("L", (280, 280), 255))
    blank_response = client.post("/api/predict", json={"image": blank})
    assert blank_response.status_code == 400, "El lienzo vacío debe producir error 400."
    invalid_response = client.post("/api/predict", json={"image": "no-es-una-imagen"})
    assert invalid_response.status_code == 400, "La imagen inválida debe producir error 400."

accuracy_api = correct / len(cases)
unique_predictions = sorted(set(predictions))
print(f"Exactitud de API en {len(cases)} ejemplos MNIST transformados: {correct}/{len(cases)} = {accuracy_api:.1%}")
print("Dígitos que predijo la API:", unique_predictions)
assert len(unique_predictions) >= 8, f"El servicio sigue concentrándose en muy pocas clases: {unique_predictions}"
assert accuracy_api >= 0.80, f"La prueba de extremo a extremo obtuvo solo {accuracy_api:.1%}; revisa la transformación de imágenes."
print("Pruebas de API terminadas correctamente.")

## 6. Iniciar la aplicación nueva

Se usa el puerto **5001**, diferente del proyecto anterior. Así se evita que el navegador siga consultando una instancia antigua del SVM en `http://127.0.0.1:5000`.

In [ ]:
import os
import subprocess
import sys
import time
import urllib.error
import urllib.request

PORT = 5001
URL = f"http://127.0.0.1:{PORT}"

# Detener solo el proceso iniciado desde esta sesión de notebook, si existe.
if "cnn_server_process" in globals() and cnn_server_process.poll() is None:
    cnn_server_process.terminate()
    try:
        cnn_server_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        cnn_server_process.kill()
        cnn_server_process.wait(timeout=5)

env = os.environ.copy()
env["DIGIT_PORT"] = str(PORT)
cnn_server_process = subprocess.Popen(
    [sys.executable, "app.py"],
    cwd=str(BACKEND_DIR),
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(40):
    if cnn_server_process.poll() is not None:
        raise RuntimeError(
            f"El servidor CNN se cerró. Comprueba si el puerto {PORT} ya está ocupado "
            "y que el modelo digit_cnn.keras exista."
        )
    try:
        with urllib.request.urlopen(URL + "/api/health", timeout=1) as response:
            info = __import__("json").load(response)
            if response.status == 200 and info.get("model") == "CNN-MNIST":
                print("Aplicación CNN activa:", URL)
                print("Modelo cargado desde:", info.get("model_path"))
                break
    except (urllib.error.URLError, TimeoutError, ValueError):
        time.sleep(0.5)
else:
    cnn_server_process.terminate()
    raise TimeoutError(f"El servicio no respondió correctamente en {URL}.")

## 7. Detener el servidor

Ejecuta esta celda al terminar de probar la aplicación.

In [ ]:
if "cnn_server_process" in globals() and cnn_server_process.poll() is None:
    cnn_server_process.terminate()
    try:
        cnn_server_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        cnn_server_process.kill()
        cnn_server_process.wait(timeout=5)
    print("Servidor CNN detenido.")
else:
    print("No hay un servidor CNN activo desde esta sesión.")